In [1]:
import pandas as pd
import numpy as np
import joblib
import tensorflow as tf
import ast
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Embedding, GRU, LSTM, Dense, Dropout, 
                                     Bidirectional, Conv1D, GlobalMaxPooling1D, 
                                     Flatten, Concatenate)
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
# Load and clean data
df = pd.read_json("../merged_cleaned_data.json")
df.dropna(subset=["title", "super_category","category_1","category_2", "Bow", "weighted_rating_class"], inplace=True)
df = df.sample(n=1000000, random_state=6).reset_index(drop=True)

C:\Users\kurt_\anaconda3\envs\pytorch-transformer\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [ ]:
# Process labels
def list_to_str(x):
    if isinstance(x, str):
        try:
            lst = ast.literal_eval(x)
            return ' '.join(sorted(set(lst))) if isinstance(lst, list) else str(lst)
        except:
            return str(x)
    return str(x)

df["super_str"] = df["super_category"].apply(list_to_str)
df["cat1_str"] = df["category_1"].apply(list_to_str)
df["cat2_str"] = df["category_2"].apply(list_to_str)

# -------------------------------
# 2. Hiyerarşik label encoding
# -------------------------------
le_super = LabelEncoder()
df["y_super"] = le_super.fit_transform(df["super_str"])
y_super = to_categorical(df["y_super"])


cat1_mapping = {}
y_cat1_list = []
for super_label in df["super_str"].unique():
    sub_df = df[df["super_str"] == super_label]
    le = LabelEncoder()
    sub_labels = le.fit_transform(sub_df["cat1_str"])
    cat1_mapping[super_label] = le
    y_cat1_list.extend(sub_labels)
df["y_cat1"] = y_cat1_list
y_cat1 = to_categorical(df["y_cat1"])


cat2_mapping = {}
y_cat2_list = []
for super_label in df["super_str"].unique():
    sub_df1 = df[df["super_str"] == super_label]
    cat2_mapping[super_label] = {}
    for cat1_label in sub_df1["cat1_str"].unique():
        sub_df2 = sub_df1[sub_df1["cat1_str"] == cat1_label]
        le = LabelEncoder()
        sub_labels = le.fit_transform(sub_df2["cat2_str"])
        cat2_mapping[super_label][cat1_label] = le
        y_cat2_list.extend(sub_labels)
df["y_cat2"] = y_cat2_list
y_cat2 = to_categorical(df["y_cat2"])

In [3]:
# -------------------------------
# 3. Metin temizleme ve tokenization
# -------------------------------
def clean_text(text):
    if isinstance(text, str):
        words = text.lower().strip().split()
        return ' '.join(sorted(set(words)))
    return ""

df["clean_text"] = df["Bow"].apply(clean_text)
tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")
tokenizer.fit_on_texts(df["clean_text"])
seq = tokenizer.texts_to_sequences(df["clean_text"])
padded_seq = pad_sequences(seq, maxlen=40, padding="post")

# -------------------------------
# 4. Train-test split
# -------------------------------
X_train_text, X_test_text, y_train_super, y_test_super, y_train_cat1, y_test_cat1, y_train_cat2, y_test_cat2 = train_test_split(
    padded_seq, y_super, y_cat1, y_cat2, test_size=0.3, random_state=42, stratify=np.argmax(y_super, axis=1)
)

In [4]:
# -------------------------------
# 5. Model mimarisi
# -------------------------------
def build_model(model_type="bigru"):
    input_text = Input(shape=(40,))
    x = Embedding(input_dim=5000, output_dim=300)(input_text)
    
    if model_type == "gru":
        x = GRU(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.3)(x)
        x = GRU(128, dropout=0.3, recurrent_dropout=0.3)(x)
    elif model_type == "lstm":
        x = LSTM(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.3)(x)
        x = LSTM(128, dropout=0.3, recurrent_dropout=0.3)(x)
    elif model_type == "bigru":
        x = Bidirectional(GRU(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.3))(x)
        x = GlobalMaxPooling1D()(x)
    elif model_type == "bilstm":
        x = Bidirectional(LSTM(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.3))(x)
        x = GlobalMaxPooling1D()(x)
    elif model_type == "cnn":
        x = Conv1D(128, 5, activation='relu')(x)
        x = GlobalMaxPooling1D()(x)
    elif model_type == "mlp":
        x = Flatten()(x)
    else:
        raise ValueError("Invalid model type")

    z = Dropout(0.5)(x)
    z = Dense(256, activation='relu')(z)

    output_super = Dense(y_super.shape[1], activation='softmax', name="super_output")(z)
    output_cat1 = Dense(y_cat1.shape[1], activation='softmax', name="cat1_output")(z)
    output_cat2 = Dense(y_cat2.shape[1], activation='softmax', name="cat2_output")(z)

    model = Model(inputs=input_text, outputs=[output_super, output_cat1, output_cat2])
    model.compile(optimizer=Adam(0.00025),
                  loss="categorical_crossentropy",
                  metrics=["accuracy"])
    return model

In [5]:
# -------------------------------
# 6. Eğitim fonksiyonu
# -------------------------------
def train(model_type="bigru"):
    model = build_model(model_type)
    history = model.fit(
        X_train_text,
        [y_train_super, y_train_cat1, y_train_cat2],
        validation_data=(X_test_text, [y_test_super, y_test_cat1, y_test_cat2]),
        epochs=5,
        batch_size=32,
        callbacks=[EarlyStopping(patience=3, restore_best_weights=True)]
    )
    model.save(f"{model_type}_multioutput_model.keras")
    joblib.dump(tokenizer, "tokenizer.pkl")
    joblib.dump(le_super, "le_super.pkl")
    joblib.dump(cat1_mapping, "cat1_mapping.pkl")
    joblib.dump(cat2_mapping, "cat2_mapping.pkl")
    return history

In [ ]:
# -------------------------------
# 7. Hiyerarşik öneri fonksiyonu
# -------------------------------
def get_recommendations(user_text, model_type="bigru", top_k=5):
    model = tf.keras.models.load_model(f"{model_type}_multioutput_model.keras")
    tokenizer = joblib.load("tokenizer.pkl")
    le_super = joblib.load("le_super.pkl")
    cat1_mapping = joblib.load("cat1_mapping.pkl")
    cat2_mapping = joblib.load("cat2_mapping.pkl")

    clean_input = ' '.join(sorted(set(user_text.lower().strip().split())))
    seq = tokenizer.texts_to_sequences([clean_input])
    padded = pad_sequences(seq, maxlen=40, padding="post")

    preds_super, preds_cat1, preds_cat2 = model.predict(padded, verbose=0)
    pred_super_idx = np.argmax(preds_super)
    pred_super = le_super.inverse_transform([pred_super_idx])[0]


    le_cat1 = cat1_mapping[pred_super]
    pred_cat1_idx = np.argmax(preds_cat1)
    pred_cat1 = le_cat1.inverse_transform([pred_cat1_idx])[0]


    le_cat2 = cat2_mapping[pred_super][pred_cat1]
    pred_cat2_idx = np.argmax(preds_cat2)
    pred_cat2 = le_cat2.inverse_transform([pred_cat2_idx])[0]


    filtered_df = df[
        (df["super_str"] == pred_super) &
        (df["cat1_str"] == pred_cat1) &
        (df["cat2_str"] == pred_cat2)
    ].copy()
    if len(filtered_df) == 0:
        return (pred_super, pred_cat1, pred_cat2), None


    embed_model = Model(inputs=model.inputs, outputs=model.layers[-3].output)
    user_embedding = embed_model.predict(padded, verbose=0)
    seqs = tokenizer.texts_to_sequences(filtered_df["clean_text"])
    padded_seqs = pad_sequences(seqs, maxlen=40, padding="post")
    prod_embeddings = embed_model.predict(padded_seqs, verbose=0)

    similarities = cosine_similarity(user_embedding, prod_embeddings)[0]
    filtered_df["similarity"] = similarities
    top_products = filtered_df.sort_values(by="similarity", ascending=False).head(top_k)

    return (pred_super, pred_cat1, pred_cat2), top_products[["title", "price", "rating", "product_url", "similarity"]]


In [7]:
# -------------------------------
# 8. Örnek kullanım
# -------------------------------
selected_model_type = "cnn"
train(selected_model_type)

sample_input = "Araç torpido üstü aksesuarı"
(pred_super, pred_cat1, pred_cat2), results = get_recommendations(
    user_text=sample_input,
    model_type=selected_model_type,
    top_k=5
)

print("Predicted Supercategory:", pred_super)
print("Predicted Category_1:", pred_cat1)
print("Predicted Category_2:", pred_cat2)

if results is not None:
    print("Top recommended products:")
    print(results.to_string(index=False))
else:
    print("No products found in this supercategory")

Epoch 1/5
21875/21875 [==============================] - 118s 5ms/step - loss: 4.1998 - super_output_loss: 0.0281 - cat1_output_loss: 1.6841 - cat2_output_loss: 2.4876 - super_output_accuracy: 0.9937 - cat1_output_accuracy: 0.3758 - cat2_output_accuracy: 0.1733 - val_loss: 4.1477 - val_super_output_loss: 1.2918e-04 - val_cat1_output_loss: 1.6719 - val_cat2_output_loss: 2.4757 - val_super_output_accuracy: 1.0000 - val_cat1_output_accuracy: 0.3803 - val_cat2_output_accuracy: 0.1747
Epoch 2/5
21875/21875 [==============================] - 121s 6ms/step - loss: 4.1500 - super_output_loss: 0.0018 - cat1_output_loss: 1.6729 - cat2_output_loss: 2.4753 - super_output_accuracy: 0.9998 - cat1_output_accuracy: 0.3796 - cat2_output_accuracy: 0.1760 - val_loss: 4.1466 - val_super_output_loss: 1.2206e-04 - val_cat1_output_loss: 1.6707 - val_cat2_output_loss: 2.4757 - val_super_output_accuracy: 1.0000 - val_cat1_output_accuracy: 0.3803 - val_cat2_output_accuracy: 0.1746
Epoch 3/5
21875/21875 [=======

In [8]:
sample_input = "Kadın kot pantolon lacivert"
(pred_super, pred_cat1, pred_cat2), results = get_recommendations(
    user_text=sample_input,
    model_type=selected_model_type,
    top_k=5
)

print("Predicted Supercategory:", pred_super)
print("Predicted Category_1:", pred_cat1)
print("Predicted Category_2:", pred_cat2)

if results is not None:
    print("Top recommended products:")
    print(results.to_string(index=False))
else:
    print("No products found in this supercategory")

Predicted Supercategory: moda
Predicted Category_1: giyim ayakkabi
Predicted Category_2: kadin
Top recommended products:
                                                                                                                                                   title   price  rating                                                                                                                                                                                                             product_url  similarity
                                                                            tekno trust turuncu bay kadin yagmurluk kaliteli eva kumas yagmurluk outdoor 1191.00     0.0                                                                                               https://www.hepsiburada.com/tekno-trust-turuncu-bay-kadin-yagmurluk-kaliteli-eva-kumas-yagmurluk-outdoor-pm-HBC0000625F7O    0.708897
                       ozel spor yaris motosikletleri kkawasakis bandana boyun isitic

In [9]:
sample_input = "Araç içi aksesuarı ışıklandırma seti"
(pred_super, pred_cat1, pred_cat2), results = get_recommendations(
    user_text=sample_input,
    model_type=selected_model_type,
    top_k=5
)

print("Predicted Supercategory:", pred_super)
print("Predicted Category_1:", pred_cat1)
print("Predicted Category_2:", pred_cat2)

if results is not None:
    print("Top recommended products:")
    print(results.to_string(index=False))
else:
    print("No products found in this supercategory")

Predicted Supercategory: moda
Predicted Category_1: giyim ayakkabi
Predicted Category_2: kadin
Top recommended products:
                                                                                                                            title   price  rating                                                                                                                                                                              product_url  similarity
ozel spor yaris motosikletleri kkawasakis bandana boyun isiticisi erkek kadin kis kayak tupu atki tozluk yuz ortusu yurt disindan 1113.56     0.0 https://www.hepsiburada.com/de-beers-ozel-spor-yaris-motosikletleri-k-kawasakis-bandana-boyun-isiticisi-erkek-kadin-kis-kayak-tupu-atki-tozluk-yuz-ortusu-yurt-disindan-pm-HBC00007VW45V    0.739206
ozel spor yaris motosikletleri kkawasakis bandana boyun isiticisi erkek kadin kis kayak tupu atki tozluk yuz ortusu yurt disindan 1113.56     0.0 https://www.hepsiburada.com/de-beers-ozel-spor-

In [10]:
sample_input = "Iphone cep telefonu aksesuarı telefon tutacağı"
(pred_super, pred_cat1, pred_cat2), results = get_recommendations(
    user_text=sample_input,
    model_type=selected_model_type,
    top_k=5
)

print("Predicted Supercategory:", pred_super)
print("Predicted Category_1:", pred_cat1)
print("Predicted Category_2:", pred_cat2)

if results is not None:
    print("Top recommended products:")
    print(results.to_string(index=False))
else:
    print("No products found in this supercategory")

Predicted Supercategory: elektronik
Predicted Category_1: elektronigi
Predicted Category_2: isitma sogutma
Top recommended products:
                                                                      title    price  rating                                                                                                                                     product_url  similarity
                           emir kamp sobasi tuplu soba nurgaz tup ustu soba   474.32     4.3                                            https://www.hepsiburada.com/emir-kamp-sobasi-tuplu-soba-ng-309-nurgaz-tup-ustu-soba-pm-HBC0000492R1K    0.999998
                                kamp sobasi tuplu soba nurgaz tup ustu soba   572.00     4.2                                        https://www.hepsiburada.com/bba-home-kamp-sobasi-tuplu-soba-ng-309-nurgaz-tup-ustu-soba-pm-HBC0000492R2Y    0.999997
izmir siyah motorlu kizil kanatli aydinlatmasiz kumandali tavan vantilatoru 17884.00     0.0 https://www.hepsiburada.com

In [11]:
sample_input = "Nvidia 3060 ekran kartlı notebook bilgisayar lenovo"
(pred_super, pred_cat1, pred_cat2), results = get_recommendations(
    user_text=sample_input,
    model_type=selected_model_type,
    top_k=5
)

print("Predicted Supercategory:", pred_super)
print("Predicted Category_1:", pred_cat1)
print("Predicted Category_2:", pred_cat2)

if results is not None:
    print("Top recommended products:")
    print(results.to_string(index=False))
else:
    print("No products found in this supercategory")

Predicted Supercategory: elektronik
Predicted Category_1: elektronigi
Predicted Category_2: ses goruntu sistemleri
Top recommended products:
                                                                                                                                               title   price  rating                                                                                                                                                                                                   product_url  similarity
                                    【hazir stok】 adet sevimli makaron duzeltme bandi morandi renkli ogrenci kirtasiye ofis malzemeleri yurt disindan  781.20     0.0                                     https://www.hepsiburada.com/betty-becky-hazir-stok-6-adet-sevimli-makaron-duzeltme-bandi-morandi-renkli-ogrenci-kirtasiye-ofis-malzemeleri-yurt-disindan-pm-HBC00005FPMB1    0.987237
                                                     bluetooth kulaklik temizleme kalemi klav

In [12]:
sample_input = "Erkek alt giyim pantolon"
(pred_super, pred_cat1, pred_cat2), results = get_recommendations(
    user_text=sample_input,
    model_type=selected_model_type,
    top_k=5
)

print("Predicted Supercategory:", pred_super)
print("Predicted Category_1:", pred_cat1)
print("Predicted Category_2:", pred_cat2)

if results is not None:
    print("Top recommended products:")
    print(results.to_string(index=False))
else:
    print("No products found in this supercategory")

Predicted Supercategory: moda
Predicted Category_1: giyim ayakkabi
Predicted Category_2: kadin
Top recommended products:
                                                                                                                                                   title   price  rating                                                                                                                                                                                                             product_url  similarity
black peach heart style winter cow print underarm bags for women soft pluk leopard small shoulder bags female warm fluffy tote bags bolsas yurt disindan 2737.60     0.0 https://www.hepsiburada.com/qiuming-shop-black-peach-heart-style-2021-winter-cow-print-underarm-bags-for-women-soft-pluk-leopard-small-shoulder-bags-female-warm-fluffy-tote-bags-bolsas-yurt-disindan-pm-HBC00005Y9EU3    0.852291
                       ozel spor yaris motosikletleri kkawasakis bandana boyun isitic

In [15]:
sample_input = "Oyuncu bilgisayarı"
(pred_super, pred_cat1, pred_cat2), results = get_recommendations(
    user_text=sample_input,
    model_type=selected_model_type,
    top_k=5
)

print("Predicted Supercategory:", pred_super)
print("Predicted Category_1:", pred_cat1)
print("Predicted Category_2:", pred_cat2)

if results is not None:
    print("Top recommended products:")
    print(results.to_string(index=False))
else:
    print("No products found in this supercategory")

Predicted Supercategory: moda
Predicted Category_1: giyim ayakkabi
Predicted Category_2: kadin
Top recommended products:
                                                                                                                            title   price  rating                                                                                                                                                                              product_url  similarity
                                                               desenli kahve krem renkli sik kalemlik ofis okul kullanimina uygun  764.00     0.0                                                                    https://www.hepsiburada.com/desenli-kahve-ve-krem-renkli-sik-kalemlik-ofis-ve-okul-kullanimina-uygun-pm-HBC000099S0ON    0.707336
ozel spor yaris motosikletleri kkawasakis bandana boyun isiticisi erkek kadin kis kayak tupu atki tozluk yuz ortusu yurt disindan 1113.56     0.0 https://www.hepsiburada.com/de-beers-ozel-spor-